# PRD error analysis

Ноутбук загружает artifacts финального PRD run из MLflow: `error_analysis.csv`, `error_analysis.md`, `robustness_report.csv`.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MLFLOW_TRACKING_URI", "http://localhost:5000")
os.environ.setdefault("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

In [ ]:
import mlflow
import pandas as pd
from IPython.display import Markdown, display

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("movie-recs-prd")
assert experiment is not None, "Experiment movie-recs-prd not found"
runs = client.search_runs([experiment.experiment_id], filter_string="tags.stage = 'PRD'", order_by=["attributes.start_time DESC"], max_results=1)
assert runs, "No PRD runs found"
run = runs[0]
run.info.run_id

In [ ]:
from mlflow.artifacts import download_artifacts

artifact_root = download_artifacts(run_id=run.info.run_id)
artifact_root = Path(artifact_root)
error_csv = next(artifact_root.rglob("error_analysis.csv"))
robustness_csv = next(artifact_root.rglob("robustness_report.csv"))
error_report = next(artifact_root.rglob("error_analysis.md"))

errors = pd.read_csv(error_csv)
robustness = pd.read_csv(robustness_csv)
display(Markdown(error_report.read_text(encoding="utf-8")))
display(errors.head(20))
display(robustness)